In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
from scipy import signal
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
import joblib
from sklearn.model_selection import StratifiedKFold, cross_val_score

In [ ]:

def getData_half(cfileName):
    """
    Reads only the first half of the complex samples from a .cfile.
    The file is assumed to contain interleaved float32 I and Q samples.
    """
    # Get file size in bytes
    file_size = os.path.getsize(cfileName)
    # Determine total number of float32 samples in the file
    num_floats = file_size // 4
    # Each complex sample requires 2 floats.
    # The full complex array would have num_floats//2 elements.
    # To get only half of that, we need to read:
    count = (num_floats // 4) * 2  # (num_floats//4 complex samples)
    
    # Read only 'count' float32 numbers from the file
    data = np.fromfile(cfileName, dtype="float32", count=count)
    
    # Form complex samples: every pair (I, Q)
    data_complex = data[0::2] + 1j * data[1::2]
    
    return data_complex



In [ ]:
# number of samples per class
num_samp_per_class = 10000

# FFT size for the STFT operation (which is same as the feature vector size)
fft_size = feature_vector_size = 2048

# number of overlapping samples for the STFT operation
fft_overlap = 256

In [ ]:
class_label = 0

In [ ]:
data = getData_half("/media/rasi/Other/data/API-Security/hypothesis1/autheticated_firemware/original")
print(data.shape)

In [ ]:
f, t, Zxx = signal.stft(data, fs=20e6, nperseg=fft_size, noverlap=fft_overlap)
print(len(f))
print(len(t))
print(Zxx.shape)

In [ ]:
del data

Zxx = Zxx.transpose()

X = Zxx[:num_samp_per_class]
y = np.full(num_samp_per_class, class_label)
del Zxx

print(X.shape)
print(y.shape)

In [ ]:
class_label = 1 #remove random API

In [ ]:
data = getData_half("/media/rasi/Other/data/API-Security/hypothesis1/unauthenticated_firemware/ping_middle")
print(data.shape)

In [ ]:
f, t, Zxx = signal.stft(data, fs=20e6, nperseg=fft_size, noverlap=fft_overlap)
print(len(f))
print(len(t))
print(Zxx.shape)

In [ ]:
del data

Zxx = Zxx.transpose()

X = np.concatenate((X, Zxx[:num_samp_per_class]), axis=0) 
y = np.concatenate((y, np.full(num_samp_per_class, class_label)), axis=0)
del Zxx

print(X.shape)
print(y.shape)

In [ ]:
X = np.abs(X)
scaler = StandardScaler()
scaler.fit(X)
X = scaler.transform(X)

In [ ]:
# Split data to train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=0)

In [ ]:
clf = SVC()
clf.fit(X_train, y_train)
# Save the model
joblib.dump(clf, "/media/rasi/Other/data/model/svm_ping_last.pkl")  # Saves as a .pkl file
print("Model saved successfully!")

In [ ]:

clf = joblib.load("/media/rasi/Other/data/model/svm_ping_last.pkl")
y_pred = clf.predict (X_test)
# Evaluate accuracy
print("SVM Accuracy:", accuracy_score(y_test, y_pred))

In [ ]:
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

In [ ]:
n_folds = 5
skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)

# Perform cross-validation
cv_scores = cross_val_score(clf, X, y, cv=skf, scoring="accuracy")

# Print results
print(f"Cross-validation scores: {cv_scores}")
print(f"Mean Accuracy: {cv_scores.mean():.4f} ? {cv_scores.std():.4f}")

In [ ]:
plt.figure(figsize=(8,6))
plt.plot(range(1, n_folds+1), cv_scores, marker='o', linestyle='-', color='blue')
plt.title(f"Accuracy per Fold in {n_folds}-Fold Cross-Validation")
plt.xlabel("Fold Number")
plt.ylabel("Accuracy")
plt.ylim(0, 1)  # Accuracy range (0 to 1)
plt.grid(True)
plt.show()